# 第7章 数据输入输出 · 课堂代码

> 本 notebook 与课件《Python金融数据分析 · 第7章 数据输入输出》配套。
> 内容改编自《Python金融大数据分析（第2版）》第9章。

**使用说明**
- 继续用第6章的真实数据 `eod_data.csv`（本目录下已有）；
- 本章会生成若干输出文件（notes.txt、stats.csv、data.h5 等），边学边看。

In [ ]:
import numpy as np
import pandas as pd

## 1. Python基本文件操作

`"w"` 写（覆盖）、`"r"` 读（默认）、`"a"` 追加；`with` 语句用完自动关闭文件。
中文内容务必加 `encoding="utf-8"`。

In [ ]:
# 写文件：把分析结论记下来
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("第6章结论：SPY存在明显厚尾\n")
    f.write("beta(AAPL) = 0.96\n")

# 读文件：逐行看回来
with open("notes.txt", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

## 2. CSV：金融数据的通用语言

### 2.1 读CSV：三件套参数

In [ ]:
df = pd.read_csv("eod_data.csv",
                 index_col=0,       # 第0列作索引
                 parse_dates=True)  # 索引解析为日期
df = df.dropna(subset=["SPY", "AAPL.O"])
df.head(3)

### 2.2 写CSV：把计算结果保存下来

`to_csv` 与 `read_csv` 是一对"镜像"操作。下次打开 notebook 直接读回，不用重算。

In [ ]:
rets = np.log(df / df.shift(1))

stats = pd.DataFrame({
    "年化收益": rets.mean() * 252,
    "年化波动率": rets.std() * np.sqrt(252)})
stats.to_csv("stats.csv")            # 默认带索引列
stats.to_csv("stats_no_index.csv",
             index=False)            # 不写索引
stats.head(3)

### 2.3 读回验证

写完立即读回验证是好习惯（"往返一致性"）。

In [ ]:
back = pd.read_csv("stats.csv", index_col=0)
print(back.head(3))
back.equals(stats)   # True：一来一回数据没变

## 3. Excel：报告和协作的配套

> 注意：pandas 读写 `.xlsx` 需要引擎库 `openpyxl`。
> 若本机未安装，先在命令行执行一次：`pip install openpyxl`

写法如下（安装后即可运行）：

```python
df.to_excel("data.xlsx", sheet_name="价格")
stats.to_excel("data.xlsx", sheet_name="统计")   # 第二个表

pd.read_excel("data.xlsx", sheet_name="统计", index_col=0)
```

**CSV vs Excel 经验法则**：中间数据、程序间交换 → CSV/HDF5；
最终交付给业务同事的报告 → Excel。

## 4. HDF5：大数据的高速通道

HDF5 是二进制格式：读写快、可压缩，适合大规模数值数据。
`key` 相当于文件里的"表名"，一个 `.h5` 文件可存多张表。

In [ ]:
# 写：complevel 开启压缩（引擎是PyTables）
df.to_hdf("data.h5", key="prices",
          mode="w", complevel=9, complib="zlib")

# 读：一行读回完整DataFrame，日期索引原样保留
back_h5 = pd.read_hdf("data.h5", "prices")
back_h5.equals(df)     # True：一来一回数据没变

In [ ]:
# 实测两种格式的文件大小
import os
print("CSV :", round(os.path.getsize("eod_data.csv") / 1024), "KB")
print("HDF5:", round(os.path.getsize("data.h5") / 1024), "KB")

## 5. 综合案例：一条完整的分析流水线

读入 → 计算 → 保存：把前六章的技能串成真实工作流。

In [ ]:
# 1. 读入原始数据（第6章已熟悉）
raw = pd.read_csv("eod_data.csv", index_col=0,
                  parse_dates=True).dropna(subset=["SPY"])

# 2. 计算：对数收益率与滚动波动率
rets = np.log(raw / raw.shift(1))
vol = rets["SPY"].rolling(21).std() * np.sqrt(252)

# 3. 保存中间结果：明天不用重算
vol.dropna().to_csv("spy_vol21.csv", header=True)

# 4. 读回抽查
pd.read_csv("spy_vol21.csv", index_col=0, parse_dates=True).tail(3)

## 6. 课后练习

1. 把第6章练习中算出的 MSFT beta 与 AAPL beta 写入一个 CSV 文件，再读回并打印。
2. 用 `with open` 写一个文本文件，记录你对"厚尾现象"的三句话总结。
3. 选做：对 `stats.csv` 分别用 `index=True` 与 `index=False` 写出，
   用记事本打开比较两个文件的差别。

<details>
<summary>练习1 参考答案（点击展开）</summary>

```python
rets = np.log(df / df.shift(1))
x = rets[".SPX"].dropna()
betas = pd.DataFrame({
    "beta": [np.polyfit(x, rets["AAPL.O"].dropna(), 1)[0],
             np.polyfit(x, rets["MSFT.O"].dropna(), 1)[0]]},
    index=["AAPL", "MSFT"])
betas.to_csv("betas.csv")
print(pd.read_csv("betas.csv", index_col=0))
```

</details>

<details>
<summary>练习2 参考答案（点击展开）</summary>

```python
with open("fat_tail_summary.txt", "w", encoding="utf-8") as f:
    f.write("1. SPY日收益|r|>3%的天数达27天，远超正态预测\n")
    f.write("2. 极端行情往往连续出现（波动率聚集）\n")
    f.write("3. 风险管理不能只依赖均值与方差\n")

print(open("fat_tail_summary.txt", encoding="utf-8").read())
```

</details>